# Furas Data Pipeline — Silver to Gold

## Purpose
This notebook transforms the standardized Silver tender data into a curated, business-ready Gold layer for analytics and reporting.

## Input
- Unified Silver tender dataset stored using Delta Lake.

## Main Processing
- Identify technology-related tender opportunities.
- Record the reason for technology classification.
- Determine the opportunity state (Active, Expired, Closed, etc.).
- Prepare Arabic and English tender titles.
- Reuse cached translations and translate only new tenders when required.
- Standardize the final Gold structure.
- Build the Gold dimensional model (Fact, Dimensions, and Bridge tables).
- Perform final data-quality and integrity checks.

## Output
- Curated technology tender data.
- Gold Fact and Dimension tables.
- Business-ready Gold layer for analytics and Power BI reporting.

## Technologies
Azure Databricks | PySpark | Delta Lake | Unity Catalog | Azure Translator | ADLS Gen2 | Power BI

## Pipeline Position
Sources → ADLS → Bronze → Silver → Gold → Power BI

In [0]:
silver_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/"
    "tenders_unified"
)

silver_final = (
    spark.read
    .format("delta")
    .load(silver_path)
)

print("Rows:", silver_final.count())
print("Columns:", len(silver_final.columns))

In [0]:
silver_final.groupBy(
    "source",
    "is_tech"
).count().orderBy(
    "source",
    "is_tech"
).show(100, truncate=False)

In [0]:
from pyspark.sql.functions import col
silver_final.groupBy("is_tech") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

In [0]:
from pyspark.sql.functions import explode_outer, col

category_counts = (
    silver_final
    .select(
        "source",
        explode_outer("categories").alias("category")
    )
    .filter(col("category").isNotNull())
    .groupBy("source", "category")
    .count()
    .orderBy("source", col("count").desc())
)

category_counts.show(200, truncate=False)

In [0]:
from pyspark.sql.functions import count

silver_final.groupBy("source").agg(
    count("*").alias("total"),
    count("categories").alias("has_categories")
).orderBy("source").show()

In [0]:
from pyspark.sql.functions import (
    col, lower, concat_ws, coalesce, lit
)

tech_text = silver_final.withColumn(
    "classification_text",
    lower(
        concat_ws(
            " ",
            coalesce(col("tender_name"), lit("")),
            coalesce(col("description"), lit("")),
            coalesce(
                concat_ws(" ", col("categories")),
                lit("")
            )
        )
    )
)

In [0]:
tech_keywords = [
    "تقنية المعلومات",
    "تقنية",
    "برمجيات",
    "برنامج",
    "نظام",
    "أنظمة",
    "تطبيق",
    "تطبيقات",
    "شبكات",
    "شبكة",
    "أمن سيبراني",
    "الأمن السيبراني",
    "اتصالات",
    "الإتصالات",
    "حاسب",
    "حواسيب",
    "خوادم",
    "سحابي",
    "الحوسبة السحابية",
    "ذكاء اصطناعي",
    "بيانات",
    "رقمي",
    "رقمية",
    "software",
    "system",
    "application",
    "network",
    "cyber",
    "cloud",
    "data",
    "digital",
    "information technology"
]

for keyword in tech_keywords:
    n = tech_text.filter(
        col("classification_text").contains(keyword)
    ).count()

    print(keyword, "->", n)

In [0]:
# ============================================================
# STEP: Inspect only the 17 suspicious Etimad tenders
# Purpose:
# Show the tender name, category, and exact keywords that
# caused each suspicious record to match the tech classifier.
# This does NOT modify any data.
# ============================================================

from pyspark.sql import functions as F

non_tech_categories = [
    "الأنشطة الطبية",
    "أنشطة الاستشارات الادارية",
    "انشطة الاستشارات المالية",
    "تجارة الكماليات",
    "تجارة مواد البناء والأدوات الكهربائية والصحية",
    "خدمات النشر والطباعة والدعاية والاعلان والإنتاج الفني",
    "خدمات صيانة واصلاح الأجهزة والمعدات والآلات",
    "مقاولات عامة للمباني (الانشاء، الإصلاح، الهدم، الترميم)",
    "\tالتشغيل والصيانة والنظافة للمنشآت - التشغيل والصيانة\t"
]

matched_array = F.array(
    *[
        F.when(
            F.col("classification_text").contains(keyword),
            F.lit(keyword)
        )
        for keyword in tech_keywords
    ]
)

suspicious_diagnostic = (
    tech_text
    .filter(F.col("source") == "etimad")
    .filter(
        F.exists(
            F.col("categories"),
            lambda x: x.isin(non_tech_categories)
        )
    )
    .withColumn("matched_raw", matched_array)
    .withColumn(
        "matched_keywords",
        F.expr("filter(matched_raw, x -> x is not null)")
    )
    .filter(F.size("matched_keywords") > 0)
    .select(
        "tender_name",
        "description",
        "categories",
        "matched_keywords"
    )
)

display(suspicious_diagnostic)

In [0]:
# # ============================================================
# # STEP: Define clear non-tech procurement signals
# # Purpose:
# # Prevent technology words from causing false positives when
# # the actual procurement is construction, stationery,
# # building work, shades, or air-conditioning materials.
# # ============================================================

# non_tech_procurement_keywords = [
#     "قرطاسية",
#     "مظلات",
#     "أعمال ترميم",
#     "اعمال ترميم",
#     "أنشاء المكاتب",
#     "إنشاء المكاتب",
#     "مواد لوحدات تكييف",
#     "مواد تكييف"
# ]

# non_tech_pattern = "|".join(non_tech_procurement_keywords)

In [0]:
# ============================================================
# STEP: Classify technology tenders for Gold
# Purpose:
# 1. Preserve explicit technology classifications from Silver.
# 2. Preserve explicit non-tech classifications from Silver.
# 3. For sources where is_tech is NULL, use strong technology
#    signals from tender name, description, and categories.
# 4. Remove the validated non-tech procurement false positives.
# 5. Avoid the "حاسب" keyword because substring matching caused
#    false positives with words such as "المحاسبية".
# ============================================================

from pyspark.sql import functions as F


# ============================================================
# STEP: Define strong technology keywords
# Purpose:
# These keywords are used only when Silver does not already
# provide an explicit is_tech classification.
# ============================================================

strong_tech_keywords = [
    "تقنية المعلومات",
    "تقنية الاتصالات",
    "الاتصالات وتقنية المعلومات",
    "برمجيات",
    "أمن سيبراني",
    "الأمن السيبراني",
    "خوادم",
    "الحوسبة السحابية",
    "ذكاء اصطناعي",
    "حواسيب",
    "software",
    "cyber",
    "cloud"
]

strong_pattern = "|".join(strong_tech_keywords)


# ============================================================
# STEP: Define validated non-tech procurement signals
# Purpose:
# Prevent known false positives where technology terminology
# refers only to a department, building, room, or location,
# while the actual procurement itself is non-tech.
# ============================================================

non_tech_procurement_keywords = [
    "قرطاسية",
    "مظلات",
    "أعمال ترميم",
    "اعمال ترميم",
    "أنشاء المكاتب",
    "إنشاء المكاتب",
    "مواد لوحدات تكييف",
    "مواد تكييف"
]

non_tech_pattern = "|".join(non_tech_procurement_keywords)


# ============================================================
# STEP: Final technology classification
# Purpose:
# Apply Silver classifications first.
# If Silver has no classification, apply the validated
# exclusions and then use strong technology signals from the
# complete classification_text.
#
# classification_text contains:
# tender_name + description + categories
# ============================================================

gold_classified = (
    tech_text
    .withColumn(
        "is_tech_gold",

        # Keep explicit Silver technology classifications.
        F.when(
            F.col("is_tech") == True,
            F.lit(True)
        )

        # Keep explicit Silver non-tech classifications.
        .when(
            F.col("is_tech") == False,
            F.lit(False)
        )

        # Remove validated non-tech procurement false positives.
        .when(
            F.col("classification_text").rlike(non_tech_pattern),
            F.lit(False)
        )

        # For NULL Silver classifications, use strong technology
        # signals from name + description + categories.
        .when(
            F.col("classification_text").rlike(strong_pattern),
            F.lit(True)
        )

        # Everything else is classified as non-tech.
        .otherwise(
            F.lit(False)
        )
    )
)


# ============================================================
# STEP: Validate classification result
# Purpose:
# Confirm technology counts by source before continuing with
# the rest of the Gold pipeline.
# ============================================================

(
    gold_classified
    .filter(F.col("is_tech_gold") == True)
    .groupBy("source")
    .count()
    .orderBy("source")
    .show()
)

print(
    "Total tech records:",
    gold_classified
    .filter(F.col("is_tech_gold") == True)
    .count()
)

In [0]:
# # ============================================================
# # STEP: Inspect tech records lost after removing categories
# # Purpose:
# # Find records that the old classifier considered technology
# # but the new name + description classifier considers non-tech.
# # This does NOT modify any data.
# # ============================================================

# lost_after_category_removal = (
#     tech_text

#     # Only inspect records without an explicit Silver decision.
#     .filter(F.col("is_tech").isNull())

#     # Old rule: name + description + categories matched tech.
#     .filter(
#         F.col("classification_text").rlike(strong_pattern)
#     )

#     # New rule: name + description does NOT match tech.
#     .filter(
#         ~gold_classification_text.rlike(strong_pattern)
#     )

#     .select(
#         "source",
#         "tender_name",
#         "description",
#         "categories"
#     )
# )

# print(
#     "Records lost after removing categories:",
#     lost_after_category_removal.count()
# )

# display(
#     lost_after_category_removal
#     .orderBy("source", "tender_name")
# )

In [0]:
# ============================================================
# STEP: Find suspicious Etimad records after classifier fix
# Purpose:
# Find records classified as technology even though their
# source category is clearly associated with a non-tech area.
# This does NOT modify any data.
# ============================================================

from pyspark.sql import functions as F

non_tech_categories = [
    "الأنشطة الطبية",
    "أنشطة الاستشارات الادارية",
    "انشطة الاستشارات المالية",
    "تجارة الكماليات",
    "تجارة مواد البناء والأدوات الكهربائية والصحية",
    "خدمات النشر والطباعة والدعاية والاعلان والإنتاج الفني",
    "خدمات صيانة واصلاح الأجهزة والمعدات والآلات",
    "مقاولات عامة للمباني (الانشاء، الإصلاح، الهدم، الترميم)",
    "\tالتشغيل والصيانة والنظافة للمنشآت - التشغيل والصيانة\t"
]

remaining_suspicious = (
    gold_classified
    .filter(F.col("source") == "etimad")
    .filter(F.col("is_tech_gold") == True)
    .filter(
        F.exists(
            F.col("categories"),
            lambda x: x.isin(non_tech_categories)
        )
    )
    .select(
        "tender_name",
        "categories"
    )
)

print(
    "Remaining suspicious Etimad records:",
    remaining_suspicious.count()
)

display(remaining_suspicious)

In [0]:
# ============================================================
# STEP: Validate non-tech exclusions across all sources
# Purpose:
# See every record removed specifically by our new
# non-tech procurement exclusion rule.
# This does NOT modify any data.
# ============================================================

excluded_by_rule = (
    tech_text
    .filter(
        F.col("is_tech").isNull()
    )
    .filter(
        F.col("classification_text").rlike(non_tech_pattern)
    )
    .filter(
        F.col("classification_text").rlike(strong_pattern)
    )
    .select(
        "source",
        "tender_name",
        "categories"
    )
)

print(
    "Total records excluded by new rule:",
    excluded_by_rule.count()
)

display(
    excluded_by_rule.orderBy("source", "tender_name")
)

In [0]:
# ============================================================
# STEP: Final validation of technology classification
# Purpose:
# Confirm the final number of technology tenders by source
# after all classifier corrections.
# This does NOT modify Gold 
# ============================================================

(
    gold_classified
    .filter(F.col("is_tech_gold") == True)
    .groupBy("source")
    .count()
    .orderBy("source")
    .show()
)

print(
    "Final total tech records:",
    gold_classified
    .filter(F.col("is_tech_gold") == True)
    .count()
)

In [0]:
# ============================================================
# STEP: Check technology flags already provided by Silver
# Purpose:
# Determine which sources already have reliable is_tech=True
# records and which sources depend on Gold classification.
# This does NOT modify any data.
# ============================================================

from pyspark.sql import functions as F

(
    tech_text
    .groupBy("source")
    .agg(
        F.count("*").alias("total_silver"),
        F.sum(
            F.when(F.col("is_tech") == True, 1).otherwise(0)
        ).alias("silver_tech_true"),
        F.sum(
            F.when(F.col("is_tech") == False, 1).otherwise(0)
        ).alias("silver_tech_false"),
        F.sum(
            F.when(F.col("is_tech").isNull(), 1).otherwise(0)
        ).alias("silver_tech_null")
    )
    .orderBy("source")
    .show(truncate=False)
)

In [0]:
newly_classified_tech = (
    gold_classified
    .filter(
        col("is_tech").isNull() &
        (col("is_tech_gold") == True)
    )
)

print(
    "Newly classified tech tenders:",
    newly_classified_tech.count()
)

newly_classified_tech.select(
    "source",
    "tender_name",
    "categories"
).show(50, truncate=False)

In [0]:
# ============================================================
# STEP: Find Etimad records classified only because of category
# Purpose:
# Identify tenders where "تقنية المعلومات" exists in the
# category but the tender title itself has no strong tech term.
# This does NOT modify any data.
# ============================================================

from pyspark.sql import functions as F

title_text = F.lower(
    F.coalesce(F.col("tender_name"), F.lit(""))
)

category_only_it = (
    gold_classified
    .filter(F.col("source") == "etimad")
    .filter(F.col("is_tech").isNull())
    .filter(F.col("is_tech_gold") == True)

    # Category says IT.
    .filter(
        F.exists(
            F.col("categories"),
            lambda x: x == "تقنية المعلومات"
        )
    )

    # But the title itself does not contain our strong signal.
    .filter(
        ~title_text.rlike(strong_pattern)
    )

    .select(
        "tender_name",
        "categories"
    )
)

print(
    "Etimad records classified from category/context only:",
    category_only_it.count()
)

display(category_only_it)

In [0]:
gold_classified.filter(
    (col("is_tech_gold") == True) &
    (col("source").isin(
        "cst",
        "geo_resources",
        "sar",
        "tahakom"
    ))
).select(
    "source",
    "tender_name",
    "description",
    "categories"
).show(50, truncate=False)

In [0]:
# ============================================================
# STEP: Inspect keywords responsible for Gold classification
# Purpose:
# Show the CURRENT strong technology keywords matched by each
# tender that actually survives the final Gold classifier.
# This is diagnostic only and does not modify any data.
# ============================================================

from pyspark.sql import functions as F

keyword_matches = gold_classified.withColumn(
    "matched_tech_keywords",
    F.array_compact(
        F.array(*[
            F.when(
                F.col("classification_text").contains(keyword),
                F.lit(keyword)
            )
            for keyword in strong_tech_keywords
        ])
    )
)


# ============================================================
# STEP: Show fallback-classified technology tenders
# Purpose:
# Inspect only records that:
# 1. Had no explicit Silver is_tech classification.
# 2. Survived the final Gold classifier as technology.
# ============================================================

(
    keyword_matches
    .filter(
        F.col("is_tech").isNull() &
        (F.col("is_tech_gold") == True)
    )
    .select(
        "source",
        "source_tender_id",
        "tender_name",
        "matched_tech_keywords"
    )
    .orderBy(
        "source",
        "tender_name"
    )
    .show(100, truncate=False)
)

In [0]:
keyword_matches.filter(
    (col("source") == "sar") &
    (col("is_tech_gold") == True)
).select(
    "tender_name",
    "matched_tech_keywords",
    "classification_text"
).show(truncate=False)

In [0]:
# ============================================================
# STEP: Validate final Gold technology classification
# Purpose:
# 1. Count technology and non-technology records.
# 2. Verify the classification distribution by source.
# 3. Use the CURRENT final classifier: gold_classified.
# ============================================================

from pyspark.sql import functions as F


# ============================================================
# STEP: Check overall classification counts
# ============================================================

gold_classified.groupBy(
    "is_tech_gold"
).count().show()


# ============================================================
# STEP: Check classification counts by source
# ============================================================

(
    gold_classified
    .groupBy(
        "source",
        "is_tech_gold"
    )
    .count()
    .orderBy(
        "source",
        "is_tech_gold"
    )
    .show(100, truncate=False)
)

In [0]:
# ============================================================
# STEP: Add technology classification reason
# Purpose:
# Explain why each record was classified as technology or
# non-technology using the CURRENT final classifier.
#
# Reasons:
# SOURCE_FLAG       -> Silver explicitly classified it as tech.
# SOURCE_NOT_TECH   -> Silver explicitly classified it non-tech.
# NON_TECH_SIGNAL   -> Removed by a validated non-tech rule.
# TECH_SIGNAL       -> Classified as tech using Gold keywords.
# NOT_TECH          -> No technology signal was found.
# ============================================================

from pyspark.sql import functions as F


gold_classified = (
    gold_classified
    .withColumn(
        "tech_classification_reason",

        # Explicit technology classification from Silver.
        F.when(
            F.col("is_tech") == True,
            F.lit("SOURCE_FLAG")
        )

        # Explicit non-technology classification from Silver.
        .when(
            F.col("is_tech") == False,
            F.lit("SOURCE_NOT_TECH")
        )

        # Record matched one of our validated non-tech exclusions.
        .when(
            F.col("classification_text").rlike(non_tech_pattern),
            F.lit("NON_TECH_SIGNAL")
        )

        # Record was classified as technology by Gold keywords.
        .when(
            F.col("classification_text").rlike(strong_pattern),
            F.lit("TECH_SIGNAL")
        )

        # No technology evidence was found.
        .otherwise(
            F.lit("NOT_TECH")
        )
    )
)


# ============================================================
# STEP: Validate classification reasons
# Purpose:
# Check that every record received a classification reason.
# ============================================================

(
    gold_classified
    .groupBy(
        "is_tech_gold",
        "tech_classification_reason"
    )
    .count()
    .orderBy(
        "is_tech_gold",
        "tech_classification_reason"
    )
    .show(truncate=False)
)

In [0]:
# ============================================================
# STEP: Inspect records removed by non-tech rules
# Purpose:
# Verify that the validated exclusion rules are not removing
# genuine technology tenders.
# This is diagnostic only and does not modify any data.
# ============================================================

(
    gold_classified
    .filter(
        F.col("tech_classification_reason") == "NON_TECH_SIGNAL"
    )
    .select(
        "source",
        "source_tender_id",
        "tender_name",
        "description",
        "categories"
    )
    .orderBy(
        "source",
        "tender_name"
    )
    .show(100, truncate=False)
)

In [0]:
# ============================================================
# STEP: Validate classification reasons by source
# Purpose:
# Show how each source's records were classified:
# - SOURCE_FLAG
# - SOURCE_NOT_TECH
# - NON_TECH_SIGNAL
# - TECH_SIGNAL
# - NOT_TECH
#
# This is diagnostic only and does not modify any data.
# ============================================================

(
    gold_classified
    .groupBy(
        "source",
        "tech_classification_reason"
    )
    .count()
    .orderBy(
        "source",
        "tech_classification_reason"
    )
    .show(100, truncate=False)
)

In [0]:
# ============================================================
# STEP: Create the final technology tender dataset
# Purpose:
# Keep only records classified as technology by the current
# validated Gold classifier.
# ============================================================

from pyspark.sql import functions as F

tech_tenders = (
    gold_classified
    .filter(F.col("is_tech_gold") == True)
)


# ============================================================
# STEP: Validate technology tender count
# Purpose:
# Confirm the final number of technology tenders.
# Expected: 2,255 records.
# ============================================================

print("Tech tenders:", tech_tenders.count())


# ============================================================
# STEP: Inspect status distribution
# Purpose:
# Review the status values among the technology tenders before
# deriving the opportunity_state field.
# ============================================================

(
    tech_tenders
    .groupBy("status")
    .count()
    .orderBy(F.col("count").desc())
    .show(truncate=False)
)

In [0]:
from pyspark.sql.functions import current_date, count, when, col

tech_tenders.select(
    count("*").alias("total"),
    count("closing_date").alias("has_closing_date")
).show()

tech_tenders.groupBy(
    when(col("closing_date").isNull(), "NO_CLOSING_DATE")
    .when(col("closing_date") < current_date(), "PAST")
    .when(col("closing_date") == current_date(), "TODAY")
    .otherwise("FUTURE")
    .alias("deadline_state")
).count().show()

In [0]:
# ============================================================
# STEP: Derive opportunity state for technology tenders
# Purpose:
# Classify each tech tender according to its official status
# and closing date while keeping historical tenders in Gold.
# ============================================================

from pyspark.sql import functions as F

gold_tenders = (
    tech_tenders

    .withColumn(
        "opportunity_state",

        # Final states reported by the source.
        F.when(
            F.col("status") == "AWARDED",
            F.lit("AWARDED")
        )
        .when(
            F.col("status") == "COMPLETED",
            F.lit("COMPLETED")
        )
        .when(
            F.col("status") == "CLOSED",
            F.lit("CLOSED")
        )
        .when(
            F.col("status") == "CANCELLED",
            F.lit("CANCELLED")
        )
        .when(
            F.col("status") == "RETURNED",
            F.lit("RETURNED")
        )

        # No closing date is available.
        .when(
            F.col("closing_date").isNull(),
            F.lit("ACTIVE_NO_DEADLINE")
        )

        # Closing date has already passed.
        .when(
            F.col("closing_date") < F.current_date(),
            F.lit("EXPIRED")
        )

        # Closing date is today or in the future.
        .otherwise(
            F.lit("ACTIVE")
        )
    )
)


# ============================================================
# STEP: Validate opportunity states
# Purpose:
# Confirm all 2,255 technology tenders received a state.
# ============================================================

print("Gold tenders:", gold_tenders.count())

(
    gold_tenders
    .groupBy("opportunity_state")
    .count()
    .orderBy(F.col("count").desc())
    .show(truncate=False)
)

In [0]:
gold_tenders.groupBy(
    "status",
    "opportunity_state"
).count().orderBy(
    "status",
    "opportunity_state"
).show(100, truncate=False)

In [0]:
from pyspark.sql import functions as F
gold_final = gold_tenders.select(
    # Identity
    "tender_key",
    "source",
    "source_tender_id",
    "reference_number",

    # Tender information
    "tender_name",
    "tender_name_en",
    "description",
    "agency_name",

    # Type
    "tender_type",

    # Status
    "status",
    "opportunity_state",

    # Dates
    "publish_date",
    "closing_date",
    "opening_date",

    # Classification
    # Replace missing categories with "Unknown" for consistent BI filtering
    F.when(
    F.col("categories").isNull() | (F.size(F.col("categories")) == 0),
    F.array(F.lit("Unknown"))
  ).otherwise(
    F.col("categories")
  ).alias("categories"),
    F.when(
    F.col("regions").isNull() | (F.size(F.col("regions")) == 0),
    F.array(F.lit("Unknown"))
   ).otherwise(
      F.col("regions")
   ).alias("regions"),

    F.when(
        F.col("tech_classification_reason").isin(
            "SOURCE_FLAG",
            "TECH_SIGNAL"
        ),
        F.lit(True)
    ).otherwise(
        F.lit(False)
    ).alias("is_tech"),

    "tech_classification_reason",

    # Financial
    "estimated_value_min",
    "estimated_value_max",
    "document_price",
    F.lit("SAR").alias("currency"),

    # Source information
    "tender_url",
    F.coalesce(
    F.col("ingestion_timestamp"),
    F.current_timestamp()
 ).alias("ingestion_timestamp")
)

print("Gold rows:", gold_final.count())
print("Gold columns:", len(gold_final.columns))

gold_final.printSchema()

In [0]:
gold_final.groupBy("currency").count().show()

In [0]:
print("Checking NEW gold_final")

gold_final.select(
    "is_tech",
    "tech_classification_reason"
).groupBy(
    "is_tech",
    "tech_classification_reason"
).count().show()

In [0]:
gold_final.printSchema()

In [0]:
from pyspark.sql.functions import count, countDistinct
gold_final.select(
    count("*").alias("rows"),
    countDistinct("tender_key").alias("unique_keys")
).show()

In [0]:
dbutils.secrets.listScopes()

In [0]:
dbutils.secrets.list("furas-secrets")

In [0]:
print("Gold rows:", gold_final.count())
print("Gold columns:", len(gold_final.columns))

In [0]:

# ============================================================
# STEP: Define translation cache location
# Purpose:
# Store and reuse translated tender titles so that existing
# translations do not require another Translator API call.
# ============================================================

translation_cache_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/"
    "tender_translations"
)

In [0]:
# ============================================================
# STEP: Compare current Gold with translation cache
# Purpose:
# Check which of the current 2,255 Gold tenders already have
# cached translations.
#
# This cell does NOT call the Translator API and does NOT
# modify any data.
# ============================================================

from pyspark.sql import functions as F


# ============================================================
# STEP: Load the persistent translation cache
# ============================================================

translation_cache = (
    spark.read
    .format("delta")
    .load(translation_cache_path)
)


# ============================================================
# STEP: Get unique tender keys from current Gold
# ============================================================

gold_keys = (
    gold_final
    .select(
        "tender_key",
        "source",
        "tender_name"
    )
    .dropDuplicates(["tender_key"])
)


# ============================================================
# STEP: Get unique keys already stored in translation cache
# ============================================================

cache_keys = (
    translation_cache
    .select("tender_key")
    .dropDuplicates(["tender_key"])
)


# ============================================================
# STEP: Find Gold tenders without a cached translation
# ============================================================

missing_translations = (
    gold_keys
    .join(
        cache_keys,
        on="tender_key",
        how="left_anti"
    )
)


# ============================================================
# STEP: Validate translation coverage
# ============================================================

print("Current Gold:", gold_keys.count())
print("Translation cache:", cache_keys.count())
print("Missing translations:", missing_translations.count())

(
    missing_translations
    .groupBy("source")
    .count()
    .orderBy("source")
    .show(truncate=False)
)

## Arabic names -> English

In [0]:
# ============================================================
# STEP: Check English title coverage
# Purpose:
# Determine how many current Gold tenders already have an
# English title and how many still require translation.
#
# This cell does NOT call the Translator API.
# ============================================================

from pyspark.sql import functions as F

gold_final.select(

    # Total number of current Gold tenders.
    F.count("*").alias("total_tenders"),

    # Tenders with no usable English title.
    F.sum(
        F.when(
            F.col("tender_name_en").isNull() |
            (F.trim(F.col("tender_name_en")) == ""),
            1
        ).otherwise(0)
    ).alias("missing_english_name"),

    # Tenders that already have an English title.
    F.sum(
        F.when(
            F.col("tender_name_en").isNotNull() &
            (F.trim(F.col("tender_name_en")) != ""),
            1
        ).otherwise(0)
    ).alias("has_english_name")

).show()

In [0]:
# ============================================================
# STEP: Check original tender title language
# Purpose:
# Identify how many Gold tender names contain Arabic text.
# This helps determine the translation direction later.


from pyspark.sql import functions as F

language_check = (
    gold_final
    .withColumn(
        "contains_arabic",
        F.col("tender_name").rlike("[\\u0600-\\u06FF]")
    )
    .groupBy("contains_arabic")
    .count()
)

display(language_check)

In [0]:
# ============================================================
# STEP: Define translation cache location
# Purpose:
# Store and reuse translated tender titles so that existing
# translations do not require another Translator API call.
# ============================================================

translation_cache_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/"
    "tender_translations"
)

In [0]:
# ============================================================
# STEP: Prepare bilingual tender title columns
# Purpose:
# Preserve the original title in the correct language column
# before joining with the persistent translation cache.
#
# Arabic original:
#   tender_name_ar = tender_name
#   tender_name_en_new = existing tender_name_en if available
#
# Non-Arabic original:
#   tender_name_en_new = tender_name
#   tender_name_ar = NULL for now
# ============================================================

from pyspark.sql import functions as F

gold_bilingual = (
    gold_final

    # Detect whether the original tender title contains Arabic.
    .withColumn(
        "original_contains_arabic",
        F.col("tender_name").rlike("[\\u0600-\\u06FF]")
    )

    # Preserve Arabic original titles.
    .withColumn(
        "tender_name_ar",
        F.when(
            F.col("original_contains_arabic"),
            F.col("tender_name")
        )
    )

    # Preserve non-Arabic original titles as English.
    # For Arabic titles, retain an existing English title
    # when the source already provided one.
    .withColumn(
        "tender_name_en_new",
        F.when(
            ~F.col("original_contains_arabic"),
            F.col("tender_name")
        ).otherwise(
            F.col("tender_name_en")
        )
    )
)


# ============================================================
# STEP: Inspect bilingual preparation
# Purpose:
# Verify that original titles were assigned to the correct
# language column before using cached translations.
# ============================================================

gold_bilingual.select(
    "tender_name",
    "original_contains_arabic",
    "tender_name_ar",
    "tender_name_en_new"
).show(10, truncate=False)

In [0]:
# ============================================================
# STEP: Load the persistent translation cache
# Purpose:
# Reuse translations from previous pipeline runs instead of
# calling the Translator API again.
# ============================================================

translation_cache = (
    spark.read
    .format("delta")
    .load(translation_cache_path)
)


# ============================================================
# STEP: Get unique tender keys already in the cache
# ============================================================

cached_keys = (
    translation_cache
    .select("tender_key")
    .dropDuplicates(["tender_key"])
)


# ============================================================
# STEP: Find new Gold tenders not present in the cache
# Purpose:
# Only these records may require a new translation API call.
# ============================================================

new_tenders = (
    gold_bilingual
    .join(
        cached_keys,
        on="tender_key",
        how="left_anti"
    )
)


# ============================================================
# STEP: Validate translation-cache coverage
# ============================================================

current_gold_count = gold_bilingual.count()
new_tender_count = new_tenders.count()

print("Current Gold tenders:", current_gold_count)
print("Already cached:", current_gold_count - new_tender_count)
print("New tenders needing translation:", new_tender_count)

### clean the untranslated data

In [0]:
# ============================================================
# STEP: Check bilingual title coverage before cache join
# Purpose:
# Measure how many Gold tenders already have Arabic and
# English titles and how many still need to be filled from
# the persistent translation cache.
#
# This cell does NOT call the Translator API.
# ============================================================

gold_bilingual.select(

    F.count("*").alias("total"),

    # Missing Arabic title.
    F.sum(
        F.when(
            F.col("tender_name_ar").isNull() |
            (F.trim(F.col("tender_name_ar")) == ""),
            1
        ).otherwise(0)
    ).alias("need_arabic_translation"),

    # Missing English title.
    F.sum(
        F.when(
            F.col("tender_name_en_new").isNull() |
            (F.trim(F.col("tender_name_en_new")) == ""),
            1
        ).otherwise(0)
    ).alias("need_english_translation"),

    # Existing Arabic title.
    F.sum(
        F.when(
            F.col("tender_name_ar").isNotNull() &
            (F.trim(F.col("tender_name_ar")) != ""),
            1
        ).otherwise(0)
    ).alias("already_has_arabic"),

    # Existing English title.
    F.sum(
        F.when(
            F.col("tender_name_en_new").isNotNull() &
            (F.trim(F.col("tender_name_en_new")) != ""),
            1
        ).otherwise(0)
    ).alias("already_has_english")

).show()

In [0]:
# dbutils.secrets.list("furas-secrets")

In [0]:
# # ============================================================
# # STEP: Configure Azure Translator and define batch translation
# # Purpose:
# # Load the Translator key securely and define a reusable
# # function for translating only new uncached tender titles.
# # ============================================================

# import requests
# import uuid

# # Load the Translator API key securely from Databricks Secrets.
# translator_key = dbutils.secrets.get(
#     scope="furas-secrets",
#     key="translator-key"
# )

# # Azure Translator endpoint.
# endpoint = "https://api.cognitive.microsofttranslator.com"


# # Translate a batch of texts from one language to another.
# def translate_batch(texts, from_lang, to_lang):
#     if not texts:
#         return []

#     url = endpoint + "/translate"

#     params = {
#         "api-version": "3.0",
#         "from": from_lang,
#         "to": to_lang
#     }

#     headers = {
#         "Ocp-Apim-Subscription-Key": translator_key,
#         "Content-Type": "application/json",
#         "X-ClientTraceId": str(uuid.uuid4())
#     }

#     body = [{"text": text} for text in texts]

#     response = requests.post(
#         url,
#         params=params,
#         headers=headers,
#         json=body,
#         timeout=60
#     )

#     response.raise_for_status()

#     return [
#         item["translations"][0]["text"]
#         for item in response.json()
#     ]


# print("Translation function ready.")

In [0]:
# ============================================================
# STEP: Identify new tenders that require translation
# Purpose:
# Split uncached tenders by translation direction so that
# only new titles are sent to Azure Translator.
# ============================================================

# New Arabic titles that need an English translation.
arabic_to_english_rows = (
    new_tenders
    .filter(
        F.col("tender_name_ar").isNotNull() &
        F.col("tender_name_en_new").isNull()
    )
    .select("tender_key", "tender_name")
    .collect()
)

# New English titles that need an Arabic translation.
english_to_arabic_rows = (
    new_tenders
    .filter(
        F.col("tender_name_ar").isNull() &
        F.col("tender_name_en_new").isNotNull()
    )
    .select("tender_key", "tender_name")
    .collect()
)

print("Arabic -> English:", len(arabic_to_english_rows))
print("English -> Arabic:", len(english_to_arabic_rows))

In [0]:
# ============================================================
# STEP: Configure Azure Translator
# Purpose:
# Load the Translator key securely and define the API endpoint.
# ============================================================

translator_key = dbutils.secrets.get(
    scope="furas-secrets",
    key="translator-key"
)

endpoint = "https://api.cognitive.microsofttranslator.com"

In [0]:
# ============================================================
# STEP: Configure robust batch translation
# Purpose:
# Define the Azure Translator function used only for new,
# uncached tenders. Retry automatically if Azure returns
# a 429 rate-limit response.
# ============================================================

import requests
import uuid
import time


# Translate a batch of titles with retry handling.
def translate_batch(texts, from_lang, to_lang, max_retries=8):
    if not texts:
        return []

    url = endpoint + "/translate"

    params = {
        "api-version": "3.0",
        "from": from_lang,
        "to": to_lang
    }

    headers = {
        "Ocp-Apim-Subscription-Key": translator_key,
        "Content-Type": "application/json",
        "X-ClientTraceId": str(uuid.uuid4())
    }

    body = [{"text": text} for text in texts]

    for attempt in range(max_retries):

        response = requests.post(
            url,
            params=params,
            headers=headers,
            json=body,
            timeout=60
        )

        # Return translations when the request succeeds.
        if response.status_code == 200:
            return [
                item["translations"][0]["text"]
                for item in response.json()
            ]

        # Wait and retry when Azure rate-limits the request.
        if response.status_code == 429:
            wait_time = 10 * (attempt + 1)

            print(
                f"Rate limit reached. "
                f"Waiting {wait_time} seconds..."
            )

            time.sleep(wait_time)
            continue

        # Raise an error for other HTTP failures.
        response.raise_for_status()

    raise Exception(
        "Translation failed after repeated rate-limit retries."
    )

In [0]:
# ============================================================
# STEP: Translate new tender rows in batches
# Purpose:
# Split new uncached tenders into small batches, send them to
# Azure Translator, and return tender_key + translated_text.
# ============================================================

def translate_rows(rows, from_lang, to_lang, batch_size=25):
    results = []

    # Process new tenders in small batches.
    for start in range(0, len(rows), batch_size):

        batch = rows[start:start + batch_size]

        # Extract tender titles from the current batch.
        texts = [
            row["tender_name"]
            for row in batch
        ]

        # Translate the batch.
        translations = translate_batch(
            texts,
            from_lang=from_lang,
            to_lang=to_lang
        )

        # Store each translation with its tender key.
        for row, translated_text in zip(batch, translations):
            results.append({
                "tender_key": row["tender_key"],
                "translated_text": translated_text
            })

        print(
            f"Translated {min(start + batch_size, len(rows))}"
            f" / {len(rows)}"
        )

        # Small pause between API batches.
        time.sleep(1)

    return results

In [0]:
# ============================================================
# STEP: Translate only new uncached tender titles
# Purpose:
# Call Azure Translator only when new tenders require a
# missing Arabic or English title.
# ============================================================

# Translate new Arabic titles to English only when needed.
if len(arabic_to_english_rows) > 0:
    arabic_to_english_translated = translate_rows(
        arabic_to_english_rows,
        from_lang="ar",
        to_lang="en"
    )
else:
    arabic_to_english_translated = []


# Translate new English titles to Arabic only when needed.
if len(english_to_arabic_rows) > 0:
    english_to_arabic_translated = translate_rows(
        english_to_arabic_rows,
        from_lang="en",
        to_lang="ar"
    )
else:
    english_to_arabic_translated = []


# Show how many new translations were created.
print(
    "Arabic -> English translated:",
    len(arabic_to_english_translated)
)

print(
    "English -> Arabic translated:",
    len(english_to_arabic_translated)
)

In [0]:
# ============================================================
# STEP: Build clean translation-cache records
# Purpose:
# Convert newly translated titles into the standard cache
# structure before saving them to the persistent Delta cache.
# ============================================================

from pyspark.sql.types import StructType, StructField, StringType

# Define the translation-cache schema.
new_translation_schema = StructType([
    StructField("tender_key", StringType(), False),
    StructField("tender_name", StringType(), True),
    StructField("tender_name_ar", StringType(), True),
    StructField("tender_name_en", StringType(), True)
])

new_translation_records = []


# Arabic -> English
for original, translated in zip(
    arabic_to_english_rows,
    arabic_to_english_translated
):
    new_translation_records.append({
        "tender_key": original["tender_key"],
        "tender_name": original["tender_name"],
        "tender_name_ar": original["tender_name"],
        "tender_name_en": translated["translated_text"]
    })


# English -> Arabic
for original, translated in zip(
    english_to_arabic_rows,
    english_to_arabic_translated
):
    new_translation_records.append({
        "tender_key": original["tender_key"],
        "tender_name": original["tender_name"],
        "tender_name_ar": translated["translated_text"],
        "tender_name_en": original["tender_name"]
    })


# Show how many new cache records were created.
print(
    "New translations to cache:",
    len(new_translation_records)
)

In [0]:
# ============================================================
# STEP: Create DataFrame for new translation-cache records
# Purpose:
# Convert newly translated records into a Spark DataFrame
# so they can be merged into the persistent translation cache.
# ============================================================

new_translation_df = spark.createDataFrame(
    new_translation_records,
    schema=new_translation_schema
)

print(
    "New translation records prepared:",
    new_translation_df.count()
)

In [0]:
# ============================================================
# STEP: Merge new translations into the persistent cache
# Purpose:
# Add translations for newly discovered tenders while keeping
# existing cached translations unchanged.
# ============================================================

from delta.tables import DeltaTable

# Only update the cache when new translations were created.
if len(new_translation_records) > 0:

    translation_cache_delta = DeltaTable.forPath(
        spark,
        translation_cache_path
    )

    (
        translation_cache_delta.alias("target")
        .merge(
            new_translation_df.alias("source"),
            "target.tender_key = source.tender_key"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(
        f"{len(new_translation_records)} new translations "
        "merged into cache."
    )

else:
    print("No new translations to merge into cache.")

In [0]:
# ============================================================
# STEP: Reload the updated translation cache
# Purpose:
# Load the latest persistent translation cache after any new
# translations have been added.
# ============================================================

translation_cache = (
    spark.read
    .format("delta")
    .load(translation_cache_path)
)

# Check the current cache size and key uniqueness.
print(
    "Translation cache records:",
    translation_cache.count()
)

print(
    "Unique tender keys:",
    translation_cache
    .select("tender_key")
    .distinct()
    .count()
)

In [0]:
# ============================================================
# STEP: Build final bilingual Gold dataset
# Purpose:
# Join the current Gold tenders with the persistent translation
# cache and produce the final Arabic and English title columns.
# ============================================================

# Reload the persistent translation cache.
translation_cache = (
    spark.read
    .format("delta")
    .load(translation_cache_path)
    .select(
        "tender_key",
        F.col("tender_name_ar").alias("cached_tender_name_ar"),
        F.col("tender_name_en").alias("cached_tender_name_en")
    )
    .dropDuplicates(["tender_key"])
)


# Join current Gold with cached translations and prefer
# original/source titles when they are already available.
gold_bilingual_joined = (
    gold_bilingual
    .join(
        translation_cache,
        on="tender_key",
        how="left"
    )
    .withColumn(
        "tender_name_ar",
        F.coalesce(
            F.col("tender_name_ar"),
            F.col("cached_tender_name_ar")
        )
    )
    .withColumn(
        "tender_name_en",
        F.coalesce(
            F.col("tender_name_en_new"),
            F.col("cached_tender_name_en")
        )
    )
)


# Keep only the final 25 Gold columns in the required order.
final_gold_column_order = [
    "tender_key",
    "source",
    "source_tender_id",
    "reference_number",
    "tender_name",
    "tender_name_en",
    "description",
    "agency_name",
    "tender_type",
    "status",
    "opportunity_state",
    "publish_date",
    "closing_date",
    "opening_date",
    "categories",
    "regions",
    "is_tech",
    "tech_classification_reason",
    "estimated_value_min",
    "estimated_value_max",
    "document_price",
    "currency",
    "tender_url",
    "ingestion_timestamp",
    "tender_name_ar"
]


gold_final_bilingual = (
    gold_bilingual_joined
    .select(*final_gold_column_order)
)


# Validate the final bilingual Gold dataset.
print("Final Gold rows:", gold_final_bilingual.count())
print("Final Gold columns:", len(gold_final_bilingual.columns))

print(
    "Missing Arabic:",
    gold_final_bilingual
    .filter(
        F.col("tender_name_ar").isNull() |
        (F.trim(F.col("tender_name_ar")) == "")
    )
    .count()
)

print(
    "Missing English:",
    gold_final_bilingual
    .filter(
        F.col("tender_name_en").isNull() |
        (F.trim(F.col("tender_name_en")) == "")
    )
    .count()
)

print(
    "Unique keys:",
    gold_final_bilingual
    .select("tender_key")
    .distinct()
    .count()
)

In [0]:
# ============================================================
# STEP: Validate Gold fact-table grain
# Purpose:
# Confirm that the final transformed Gold dataset has one
# unique row per tender before building the star schema.
# ============================================================

# Count all final Gold records.
total_records = gold_final_bilingual.count()

# Count unique tender keys.
distinct_tender_keys = (
    gold_final_bilingual
    .select("tender_key")
    .distinct()
    .count()
)

# Count records with missing tender keys.
null_tender_keys = (
    gold_final_bilingual
    .filter(F.col("tender_key").isNull())
    .count()
)

# Calculate duplicate tender keys.
duplicate_tender_keys = total_records - distinct_tender_keys

print("Total Gold records:", total_records)
print("Distinct tender keys:", distinct_tender_keys)
print("Null tender keys:", null_tender_keys)
print("Duplicate tender keys:", duplicate_tender_keys)

In [0]:
# ============================================================
# STEP: Build Gold source dimension
# Purpose:
# Create one unique dimension record for each tender source
# represented in the final Gold dataset.
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Get one row for each unique source.
dim_source = (
    gold_final_bilingual
    .select("source")
    .filter(F.col("source").isNotNull())
    .distinct()
)

# Create a stable numeric surrogate key based on source name.
source_window = Window.orderBy("source")

dim_source = (
    dim_source
    .withColumn(
        "source_key",
        F.row_number().over(source_window)
    )
    .select(
        "source_key",
        "source"
    )
)

# Validate the source dimension.
print("Source dimension rows:", dim_source.count())

display(dim_source.orderBy("source_key"))

In [0]:
# ============================================================
# STEP: Build Gold agency dimension
# Purpose:
# Create one unique dimension record for each agency represented
# in the final Gold dataset and include an Unknown agency for
# tenders where agency information is missing.
# ============================================================

# ------------------------------------------------------------
# 1. Extract distinct valid agency names.
# ------------------------------------------------------------

dim_agency_names = (
    gold_final_bilingual
    .select(
        F.trim(F.col("agency_name")).alias("agency_name")
    )
    .filter(
        F.col("agency_name").isNotNull() &
        (F.col("agency_name") != "")
    )
    .distinct()
)


# ------------------------------------------------------------
# 2. Add an Unknown agency member.
# This will be used when a tender has no agency_name.
# ------------------------------------------------------------

unknown_agency = spark.createDataFrame(
    [("Unknown",)],
    ["agency_name"]
)

dim_agency_names = (
    dim_agency_names
    .union(unknown_agency)
    .distinct()
)


# ------------------------------------------------------------
# 3. Create the surrogate agency key.
# ------------------------------------------------------------

agency_window = Window.orderBy("agency_name")

dim_agency = (
    dim_agency_names
    .withColumn(
        "agency_key",
        F.row_number().over(agency_window)
    )
    .select(
        "agency_key",
        "agency_name"
    )
)




# ------------------------------------------------------------
# 4. Validate the agency dimension.
# ------------------------------------------------------------

print("Agency dimension rows:", dim_agency.count())

display(
    dim_agency.orderBy("agency_key")
)

In [0]:
# ============================================================
# STEP: Build Gold date dimension
# Purpose:
# Create one shared calendar dimension containing all dates
# used by publish_date, closing_date, and opening_date.
# ============================================================

# ------------------------------------------------------------
# 1. Collect all dates used by the final Gold tenders.
# ------------------------------------------------------------

all_gold_dates = (
    gold_final_bilingual
    .select(F.col("publish_date").alias("full_date"))
    .union(
        gold_final_bilingual
        .select(F.col("closing_date").alias("full_date"))
    )
    .union(
        gold_final_bilingual
        .select(F.col("opening_date").alias("full_date"))
    )
    .filter(F.col("full_date").isNotNull())
    .distinct()
)


# ------------------------------------------------------------
# 2. Create calendar attributes.
# date_key uses YYYYMMDD, which is common in dimensional models.
# ------------------------------------------------------------

dim_date = (
    all_gold_dates
    .withColumn(
        "date_key",
        F.date_format("full_date", "yyyyMMdd").cast("int")
    )
    .withColumn("year", F.year("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("day_of_week", F.dayofweek("full_date"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .select(
        "date_key",
        "full_date",
        "year",
        "quarter",
        "month",
        "month_name",
        "day",
        "day_of_week",
        "day_name"
    )
)


# ------------------------------------------------------------
# Add special member for missing / unknown dates.
# -1 = Unknown / Missing
# ------------------------------------------------------------

unknown_date_row = spark.createDataFrame(
    [
        (
            -1,       # date_key
            None,     # full_date
            None,     # year
            None,     # quarter
            None,     # month
            "Unknown",# month_name
            None,     # day
            None,     # day_of_week
            "Unknown" # day_name
        )
    ],
    schema=dim_date.schema
)

dim_date = dim_date.unionByName(unknown_date_row)


# ------------------------------------------------------------
# 3. Validate the date dimension.
# ------------------------------------------------------------

print("Date dimension rows:", dim_date.count())

display(
    dim_date.orderBy("full_date")
)





In [0]:
# ============================================================
# STEP: Build Gold category dimension and bridge
# Purpose:
# Normalize the categories array into a category dimension
# and connect tenders to their categories through a bridge.
# ============================================================

# ------------------------------------------------------------
# 1. Explode the categories array.
# One tender may produce multiple tender-category rows.
# ------------------------------------------------------------

tender_categories = (
    gold_final_bilingual
    .select(
        "tender_key",
        F.explode_outer("categories").alias("category_name")
    )
    .withColumn(
        "category_name",
        F.trim(F.col("category_name"))
    )
    .filter(
        F.col("category_name").isNotNull() &
        (F.col("category_name") != "")
    )
    .dropDuplicates(
        ["tender_key", "category_name"]
    )
)


# ------------------------------------------------------------
# 2. Create the unique category dimension.
# ------------------------------------------------------------

category_names = (
    tender_categories
    .select("category_name")
    .distinct()
)

category_window = Window.orderBy("category_name")

dim_category = (
    category_names
    .withColumn(
        "category_key",
        F.row_number().over(category_window)
    )
    .select(
        "category_key",
        "category_name"
    )
)




# ------------------------------------------------------------
# 3. Create the tender-category bridge.
# Replace the category text with its category surrogate key.
# ------------------------------------------------------------

bridge_tender_category = (
    tender_categories
    .join(
        dim_category,
        on="category_name",
        how="inner"
    )
    .select(
        "tender_key",
        "category_key"
    )
    .dropDuplicates()
)


# ------------------------------------------------------------
# 4. Validate both structures.
# ------------------------------------------------------------

print(
    "Category dimension rows:",
    dim_category.count()
)

print(
    "Tender-category bridge rows:",
    bridge_tender_category.count()
)

display(
    dim_category.orderBy("category_key")
)

display(
    bridge_tender_category.limit(20)
)

In [0]:
# ============================================================
# STEP: Build Gold region dimension and bridge
# Purpose:
# Normalize the regions array into a region dimension and
# connect tenders to their regions through a bridge table.
# ============================================================

# ------------------------------------------------------------
# 1. Explode the regions array.
# One tender may be associated with multiple regions.
# ------------------------------------------------------------

tender_regions = (
    gold_final_bilingual
    .select(
        "tender_key",
        F.explode_outer("regions").alias("region_name")
    )
    .withColumn(
        "region_name",
        F.trim(F.col("region_name"))
    )
    .filter(
        F.col("region_name").isNotNull() &
        (F.col("region_name") != "")
    )
    .dropDuplicates(
        ["tender_key", "region_name"]
    )
)


# ------------------------------------------------------------
# 2. Create the unique region dimension.
# ------------------------------------------------------------

region_names = (
    tender_regions
    .select("region_name")
    .distinct()
)

region_window = Window.orderBy("region_name")

dim_region = (
    region_names
    .withColumn(
        "region_key",
        F.row_number().over(region_window)
    )
    .select(
        "region_key",
        "region_name"
    )
)


# ------------------------------------------------------------
# 3. Create the tender-region bridge.
# ------------------------------------------------------------

bridge_tender_region = (
    tender_regions
    .join(
        dim_region,
        on="region_name",
        how="inner"
    )
    .select(
        "tender_key",
        "region_key"
    )
    .dropDuplicates()
)


# ------------------------------------------------------------
# 4. Validate both structures.
# ------------------------------------------------------------

print(
    "Region dimension rows:",
    dim_region.count()
)

print(
    "Tender-region bridge rows:",
    bridge_tender_region.count()
)

display(
    dim_region.orderBy("region_key")
)

display(
    bridge_tender_region.limit(20)
)

In [0]:
# ============================================================
# STEP: Build Gold tender type dimension
# Purpose:
# Create one dimension row for each distinct standardized
# tender type used by the final Gold tender dataset.
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ------------------------------------------------------------
# 1. Clean tender type values.
# Missing or blank values become "Unknown".
# ------------------------------------------------------------

tender_type_names = (
    gold_final_bilingual
    .select(
        F.when(
            F.col("tender_type").isNull() |
            (F.trim(F.col("tender_type")) == ""),
            F.lit("Unknown")
        )
        .otherwise(F.trim(F.col("tender_type")))
        .alias("tender_type")
    )
    .distinct()
)


# ------------------------------------------------------------
# 2. Generate a surrogate key for each tender type.
# ------------------------------------------------------------

tender_type_window = Window.orderBy("tender_type")

dim_tender_type = (
    tender_type_names
    .withColumn(
        "tender_type_key",
        F.row_number().over(tender_type_window)
    )
    .select(
        "tender_type_key",
        "tender_type"
    )
)


# ------------------------------------------------------------
# 3. Validate the tender type dimension.
# ------------------------------------------------------------

print("Tender type dimension rows:", dim_tender_type.count())

display(
    dim_tender_type.orderBy("tender_type_key")
)

In [0]:
# ============================================================
# STEP: Inspect Gold status and opportunity state
# Purpose:
# Understand the relationship between standardized tender
# status and opportunity state before designing dim_status.
# ============================================================

status_analysis = (
    gold_final_bilingual
    .groupBy(
        "status",
        "opportunity_state"
    )
    .count()
    .orderBy(
        "status",
        "opportunity_state"
    )
)

print(
    "Distinct statuses:",
    gold_final_bilingual
    .select("status")
    .distinct()
    .count()
)

print(
    "Distinct opportunity states:",
    gold_final_bilingual
    .select("opportunity_state")
    .distinct()
    .count()
)

display(status_analysis)

In [0]:
# ============================================================
# STEP: Build Gold status dimension
# Purpose:
# Create one dimension row for each distinct combination of
# standardized tender status and opportunity state.
# ============================================================

# ------------------------------------------------------------
# 1. Clean status and opportunity-state values.
# Missing or blank values are represented as "UNKNOWN".
# ------------------------------------------------------------

status_combinations = (
    gold_final_bilingual
    .select(
        F.when(
            F.col("status").isNull() |
            (F.trim(F.col("status")) == ""),
            F.lit("UNKNOWN")
        )
        .otherwise(F.trim(F.col("status")))
        .alias("status"),

        F.when(
            F.col("opportunity_state").isNull() |
            (F.trim(F.col("opportunity_state")) == ""),
            F.lit("UNKNOWN")
        )
        .otherwise(F.trim(F.col("opportunity_state")))
        .alias("opportunity_state")
    )
    .distinct()
)


# ------------------------------------------------------------
# 2. Generate a surrogate key for each unique combination.
# ------------------------------------------------------------

status_window = Window.orderBy(
    "status",
    "opportunity_state"
)

dim_status = (
    status_combinations
    .withColumn(
        "status_key",
        F.row_number().over(status_window)
    )
    .select(
        "status_key",
        "status",
        "opportunity_state"
    )
)


# ------------------------------------------------------------
# 3. Validate the status dimension.
# ------------------------------------------------------------

print(
    "Status dimension rows:",
    dim_status.count()
)

display(
    dim_status.orderBy("status_key")
)

In [0]:
# # ============================================================
# # STEP: Build Tender Dimension
# # Purpose:
# # Store descriptive information about each tender separately
# # from the fact table.
# #
# # Grain:
# # One row represents one tender.
# # ============================================================

# dim_tender = (
#     gold_final_bilingual
#     .select(
#         # Business key used to connect with fact_tenders
#         F.col("tender_key"),

#         # Source identifiers
#         F.col("source_tender_id"),
#         F.col("reference_number"),

#         # Tender descriptive information
#         F.col("tender_name"),
#         F.col("tender_name_ar"),
#         F.col("tender_name_en"),
#         F.col("description"),

#         # Technology classification information
#         F.col("is_tech"),
#         F.col("tech_classification_reason"),

#         # Additional descriptive attributes
#         F.col("currency"),
#         F.col("tender_url"),

#         # Data lineage information
#         F.col("ingestion_timestamp")
#     )
#     .dropDuplicates(["tender_key"])
# )

In [0]:

# # ============================================================
# # STEP: Build Tender Dimension
# # Purpose:
# # Store descriptive information about each tender separately
# # from the fact table.
# #
# # Grain:
# # One row represents one tender.
# # ============================================================

dim_tender = (
    gold_final_bilingual
    .select(
        # Business key used to connect with fact_tenders
        F.col("tender_key"),

        # Source identifiers
        F.col("source_tender_id"),
        F.coalesce(
            F.col("reference_number"),
            F.lit("Unknown")
        ).alias("reference_number"),

        # Tender descriptive information
        F.col("tender_name"),
        F.col("tender_name_ar"),
        F.col("tender_name_en"),
        F.coalesce(
            F.col("description"),
            F.lit("Unknown")
        ).alias("description"),

        # Technology classification information
        F.col("is_tech"),
        F.col("tech_classification_reason"),

        # Additional descriptive attributes
        F.col("currency"),
        F.coalesce(
            F.col("tender_url"),
            F.lit("Unknown")
        ).alias("tender_url"),

        # Data lineage information
        F.col("ingestion_timestamp")
    )
    .dropDuplicates(["tender_key"])
)

In [0]:
# ------------------------------------------------------------
# Validate Tender Dimension
# ------------------------------------------------------------

print("Tender dimension rows:", dim_tender.count())

print(
    "Distinct tender keys:",
    dim_tender
    .select("tender_key")
    .distinct()
    .count()
)

print(
    "Missing tender keys:",
    dim_tender
    .filter(F.col("tender_key").isNull())
    .count()
)

display(dim_tender.limit(20))

In [0]:
# ============================================================
# STEP: Build Gold tender fact table
# Purpose:
# Create one row per tender and connect each tender to the
# Gold dimensions using surrogate keys.
# ============================================================

# ------------------------------------------------------------
# 1. Prepare clean dimension values in the tender dataset.
# ------------------------------------------------------------

fact_base = (
    gold_final_bilingual

    # Clean agency.
    .withColumn(
        "agency_name_clean",
        F.when(
            F.col("agency_name").isNull() |
            (F.trim(F.col("agency_name")) == ""),
            F.lit("Unknown")
        ).otherwise(F.trim(F.col("agency_name")))
    )

    # Clean tender type.
    .withColumn(
        "tender_type_clean",
        F.when(
            F.col("tender_type").isNull() |
            (F.trim(F.col("tender_type")) == ""),
            F.lit("UNKNOWN")
        ).otherwise(F.trim(F.col("tender_type")))
    )

    # Clean status.
    .withColumn(
        "status_clean",
        F.when(
            F.col("status").isNull() |
            (F.trim(F.col("status")) == ""),
            F.lit("UNKNOWN")
        ).otherwise(F.trim(F.col("status")))
    )

    # Clean opportunity state.
    .withColumn(
        "opportunity_state_clean",
        F.when(
            F.col("opportunity_state").isNull() |
            (F.trim(F.col("opportunity_state")) == ""),
            F.lit("UNKNOWN")
        ).otherwise(F.trim(F.col("opportunity_state")))
    )
)


# ------------------------------------------------------------
# 2. Join Source dimension.
# ------------------------------------------------------------

fact_joined = (
    fact_base.alias("f")
    .join(
        dim_source.alias("s"),
        F.col("f.source") == F.col("s.source"),
        "left"
    )
)


# ------------------------------------------------------------
# 3. Join Agency dimension.
# ------------------------------------------------------------

fact_joined = (
    fact_joined
    .join(
        dim_agency.alias("a"),
        F.col("f.agency_name_clean") == F.col("a.agency_name"),
        "left"
    )
)


# ------------------------------------------------------------
# 4. Join Tender Type dimension.
# ------------------------------------------------------------

fact_joined = (
    fact_joined
    .join(
        dim_tender_type.alias("tt"),
        F.col("f.tender_type_clean") == F.col("tt.tender_type"),
        "left"
    )
)


# ------------------------------------------------------------
# 5. Join Status dimension using both status fields.
# ------------------------------------------------------------

fact_joined = (
    fact_joined
    .join(
        dim_status.alias("st"),
        (
            (F.col("f.status_clean") == F.col("st.status")) &
            (
                F.col("f.opportunity_state_clean") ==
                F.col("st.opportunity_state")
            )
        ),
        "left"
    )
)


# # ------------------------------------------------------------
# # 6. Generate foreign keys for the three date roles.
# # These keys correspond to dim_date.date_key.
# # ------------------------------------------------------------

# fact_joined = (
#     fact_joined
#     .withColumn(
#         "publish_date_key",
#         F.date_format(F.col("f.publish_date"), "yyyyMMdd").cast("int")
#     )
#     .withColumn(
#         "closing_date_key",
#         F.date_format(F.col("f.closing_date"), "yyyyMMdd").cast("int")
#     )
#     .withColumn(
#         "opening_date_key",
#         F.date_format(F.col("f.opening_date"), "yyyyMMdd").cast("int")
#     )
# )


# ------------------------------------------------------------
# 6. Generate foreign keys for the three date roles.
# Missing dates use -1 = Not Available.
# These keys correspond to dim_date.date_key.
# ------------------------------------------------------------

fact_joined = (
    fact_joined

    .withColumn(
        "publish_date_key",
        F.when(
            F.col("f.publish_date").isNull(),
            F.lit(-1)
        ).otherwise(
            F.date_format(F.col("f.publish_date"), "yyyyMMdd").cast("int")
        )
    )

    .withColumn(
        "closing_date_key",
        F.when(
            F.col("f.closing_date").isNull(),
            F.lit(-1)
        ).otherwise(
            F.date_format(F.col("f.closing_date"), "yyyyMMdd").cast("int")
        )
    )

    .withColumn(
        "opening_date_key",
        F.when(
            F.col("f.opening_date").isNull(),
            F.lit(-1)
        ).otherwise(
            F.date_format(F.col("f.opening_date"), "yyyyMMdd").cast("int")
        )
    )
)


# ------------------------------------------------------------
# 7. Build the final tender fact table.
#
# Grain:
# One row represents one tender.
#
# The fact table contains:
# - Foreign keys to dimensions
# - Numeric measures for analysis
#
# Descriptive tender information is not stored here.
# Category and region use bridge tables.
# ------------------------------------------------------------

fact_tenders = (
    fact_joined
    .select(
        # Tender identifier / fact grain
        F.col("f.tender_key").alias("tender_key"),

        # Dimension foreign keys
        F.col("s.source_key").alias("source_key"),
        F.col("a.agency_key").alias("agency_key"),
        F.col("tt.tender_type_key").alias("tender_type_key"),
        F.col("st.status_key").alias("status_key"),

        # Date dimension foreign keys
        F.col("publish_date_key"),
        F.col("closing_date_key"),
        F.col("opening_date_key"),

        # Additive measure:
        # Every row represents one tender.
        F.lit(1).cast("int").alias("tender_count"),

       # Financial measures
# Missing numeric values are represented as 0 for BI analysis.
F.coalesce(
    F.col("f.estimated_value_min"),
    F.lit(0)
).alias("estimated_value_min"),

F.coalesce(
    F.col("f.estimated_value_max"),
    F.lit(0)
).alias("estimated_value_max"),

F.coalesce(
    F.col("f.document_price"),
    F.lit(0)
).alias("document_price")
    )
)


# ------------------------------------------------------------
# 8. Validate fact-table integrity.
# ------------------------------------------------------------

print("Fact tender rows:", fact_tenders.count())

print(
    "Distinct tender keys:",
    fact_tenders
    .select("tender_key")
    .distinct()
    .count()
)

print(
    "Missing source keys:",
    fact_tenders.filter(F.col("source_key").isNull()).count()
)

print(
    "Missing agency keys:",
    fact_tenders.filter(F.col("agency_key").isNull()).count()
)

print(
    "Missing tender type keys:",
    fact_tenders.filter(F.col("tender_type_key").isNull()).count()
)

print(
    "Missing status keys:",
    fact_tenders.filter(F.col("status_key").isNull()).count()
)

display(fact_tenders.limit(20))

In [0]:

print(
    "NULL publish_date_key:",
    fact_tenders.filter(F.col("publish_date_key").isNull()).count()
)

print(
    "NULL closing_date_key:",
    fact_tenders.filter(F.col("closing_date_key").isNull()).count()
)

print(
    "NULL opening_date_key:",
    fact_tenders.filter(F.col("opening_date_key").isNull()).count()
)

print(
    "-1 publish_date_key:",
    fact_tenders.filter(F.col("publish_date_key") == -1).count()
)

print(
    "-1 closing_date_key:",
    fact_tenders.filter(F.col("closing_date_key") == -1).count()
)

print(
    "-1 opening_date_key:",
    fact_tenders.filter(F.col("opening_date_key") == -1).count()
)

In [0]:
# ============================================================
# STEP: Validate Gold star-schema relationships
# Purpose:
# Confirm that fact and bridge foreign keys have matching
# records in their corresponding Gold dimensions.
# ============================================================

# ------------------------------------------------------------
# 1. Check Publish Date foreign keys.
# Null date keys are allowed, but non-null keys must exist
# in dim_date.
# ------------------------------------------------------------

invalid_publish_dates = (
    fact_tenders
    .filter(F.col("publish_date_key").isNotNull())
    .join(
        dim_date,
        F.col("publish_date_key") == F.col("date_key"),
        "left_anti"
    )
    .count()
)


# ------------------------------------------------------------
# 2. Check Closing Date foreign keys.
# ------------------------------------------------------------

invalid_closing_dates = (
    fact_tenders
    .filter(F.col("closing_date_key").isNotNull())
    .join(
        dim_date,
        F.col("closing_date_key") == F.col("date_key"),
        "left_anti"
    )
    .count()
)


# ------------------------------------------------------------
# 3. Check Opening Date foreign keys.
# ------------------------------------------------------------

invalid_opening_dates = (
    fact_tenders
    .filter(F.col("opening_date_key").isNotNull())
    .join(
        dim_date,
        F.col("opening_date_key") == F.col("date_key"),
        "left_anti"
    )
    .count()
)


# ------------------------------------------------------------
# 4. Check Category bridge keys.
# Every category key must exist in dim_category.
# ------------------------------------------------------------

invalid_category_keys = (
    bridge_tender_category
    .join(
        dim_category,
        on="category_key",
        how="left_anti"
    )
    .count()
)


# ------------------------------------------------------------
# 5. Check Region bridge keys.
# Every region key must exist in dim_region.
# ------------------------------------------------------------

invalid_region_keys = (
    bridge_tender_region
    .join(
        dim_region,
        on="region_key",
        how="left_anti"
    )
    .count()
)


# ------------------------------------------------------------
# 6. Check that bridge tender keys exist in fact_tenders.
# ------------------------------------------------------------

invalid_category_tenders = (
    bridge_tender_category
    .join(
        fact_tenders.select("tender_key"),
        on="tender_key",
        how="left_anti"
    )
    .count()
)

invalid_region_tenders = (
    bridge_tender_region
    .join(
        fact_tenders.select("tender_key"),
        on="tender_key",
        how="left_anti"
    )
    .count()
)


# ------------------------------------------------------------
# 7. Display validation results.
# All values should be 0.
# ------------------------------------------------------------

print("Invalid publish date keys:", invalid_publish_dates)
print("Invalid closing date keys:", invalid_closing_dates)
print("Invalid opening date keys:", invalid_opening_dates)

print("Invalid category keys:", invalid_category_keys)
print("Invalid region keys:", invalid_region_keys)

print("Invalid category tender keys:", invalid_category_tenders)
print("Invalid region tender keys:", invalid_region_tenders)

In [0]:
# ============================================================
# STEP: Verify Gold ADLS access
# Purpose:
# Confirm that Databricks can access the new physical Gold
# container before writing the star-schema Delta tables.
# ============================================================

gold_base_path = (
    "abfss://gold@tenderdatalake11.dfs.core.windows.net/"
)

display(
    dbutils.fs.ls(gold_base_path)
)

In [0]:
# ============================================================
# STEP: Write Gold star-schema datasets to ADLS
# Purpose:
# Persist the validated Gold fact, dimension, and bridge
# DataFrames as Delta tables in the physical Gold container.
# ============================================================

# ------------------------------------------------------------
# 1. Define the Gold tables and their storage paths.
# ------------------------------------------------------------

gold_tables = {
    "fact_tenders": fact_tenders,

    # Dimensions
    "dim_tender": dim_tender,
    "dim_source": dim_source,
    "dim_agency": dim_agency,
    "dim_date": dim_date,
    "dim_tender_type": dim_tender_type,
    "dim_status": dim_status,
    "dim_category": dim_category,
    "dim_region": dim_region,

    # Bridge tables
    "bridge_tender_category": bridge_tender_category,
    "bridge_tender_region": bridge_tender_region
}


# ------------------------------------------------------------
# 2. Write each Gold dataset in Delta format.
# Overwrite is appropriate here because we are rebuilding the
# complete analytical presentation model from validated Silver.
# overwriteSchema allows the updated fact-table structure.
# ------------------------------------------------------------

for table_name, dataframe in gold_tables.items():

    table_path = f"{gold_base_path}{table_name}/"

    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(table_path)
    )

    print(f"Written: {table_name}")

In [0]:
# ============================================================
# STEP: Verify Gold Delta datasets in ADLS
# Purpose:
# Confirm that all 11 Gold star-schema datasets were
# successfully written to the physical Gold container.
# ============================================================

gold_folders = dbutils.fs.ls(gold_base_path)

print("Gold dataset folders:", len(gold_folders))

display(gold_folders)

In [0]:
# ============================================================
# STEP: Validate categories and regions
# Purpose:
# Confirm that no Gold records have null or empty
# categories or regions after transformation.
# ============================================================

gold_final.select(
    F.sum(
        F.when(
            F.col("categories").isNull() |
            (F.size(F.col("categories")) == 0),
            1
        ).otherwise(0)
    ).alias("missing_categories"),

    F.sum(
        F.when(
            F.col("regions").isNull() |
            (F.size(F.col("regions")) == 0),
            1
        ).otherwise(0)
    ).alias("missing_regions")
).show()

In [0]:
# ============================================================
# FINAL DATA QUALITY CHECK 1: Null and completeness validation
# Purpose:
# Check important Gold fields for missing values before the
# ============================================================

required_columns = [
    "tender_key",
    "source",
    "tender_name",
    "tender_name_ar",
    "tender_name_en",
    "status",
    "opportunity_state",
    "categories",
    "regions",
    "is_tech",
    "tech_classification_reason",
    "currency",
    "ingestion_timestamp"
]

quality_checks = []

for column_name in required_columns:

    column = F.col(column_name)

    # Arrays need a different empty-value check.
    if column_name in ["categories", "regions"]:
        missing_condition = (
            column.isNull() |
            (F.size(column) == 0)
        )

    # String columns: check NULL and empty strings.
    elif column_name not in [
        "is_tech",
        "ingestion_timestamp"
    ]:
        missing_condition = (
            column.isNull() |
            (F.trim(column) == "")
        )

    # Boolean / timestamp columns: check NULL only.
    else:
        missing_condition = column.isNull()

    quality_checks.append(
        F.sum(
            F.when(missing_condition, 1).otherwise(0)
        ).alias(f"missing_{column_name}")
    )

gold_final_bilingual.select(
    F.count("*").alias("total_rows"),
    *quality_checks
).show(truncate=False)

In [0]:
# ============================================================
# FINAL DATA QUALITY CHECK 2: Gold integrity validation
# Purpose:
# Verify row uniqueness, date consistency, financial values,
# and final Gold structure.
# ============================================================

gold_final_bilingual.select(

    F.count("*").alias("total_rows"),

    F.countDistinct("tender_key").alias("unique_tender_keys"),

    F.sum(
        F.when(
            F.col("tender_key").isNull(),
            1
        ).otherwise(0)
    ).alias("null_tender_keys"),

    F.sum(
        F.when(
            F.col("estimated_value_min") < 0,
            1
        ).otherwise(0)
    ).alias("negative_min_value"),

    F.sum(
        F.when(
            F.col("estimated_value_max") < 0,
            1
        ).otherwise(0)
    ).alias("negative_max_value"),

    F.sum(
        F.when(
            F.col("document_price") < 0,
            1
        ).otherwise(0)
    ).alias("negative_document_price"),

    F.sum(
        F.when(
            F.col("estimated_value_min") >
            F.col("estimated_value_max"),
            1
        ).otherwise(0)
    ).alias("min_greater_than_max")

).show(truncate=False)

print(
    "Final Gold columns:",
    len(gold_final_bilingual.columns)
)